## Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

CATALOG = "workspace"
SCHEMA = "transfermarkt"

data = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/regression_features_final")
data.groupBy("year").count().orderBy("year").show(30)


## Log-transform monetary predictors, add age_squared, fix the minutes/appearances collinearity, flag missing club values
Money features get log1p, same shape as the target. age_squared lets the linear model represent the peak-and-decline curve the scatter plots confirmed — a parabola is linear in its coefficients even though the resulting curve isn't a straight line. avg_minutes_per_appearance replaces minutes_last_365d, which correlated at 0.966 with appearances_last_365d; the two together were carrying almost the same information, while minutes-per-appearance and appearances separately capture two genuinely different things: how big a role a player played, versus how often they featured at all. Missing club values get median-imputed in log space with an explicit flag, so the model can tell "imputed" from "genuinely small" apart.

In [0]:
data = (data
    .withColumn("log_market_value", F.log1p("market_value_in_eur"))
    .withColumn("log_from_club_value", F.log1p("from_club_value_eur"))
    .withColumn("log_to_club_value", F.log1p("to_club_value_eur"))
    .withColumn("age_squared", F.col("age_at_transfer") * F.col("age_at_transfer"))
    .withColumn("avg_minutes_per_appearance",
                F.when(F.col("appearances_last_365d") > 0,
                       F.col("minutes_last_365d") / F.col("appearances_last_365d"))
                 .otherwise(F.lit(0)))
    .withColumn("from_club_value_missing", F.col("from_club_value_eur").isNull().cast("int"))
    .withColumn("to_club_value_missing", F.col("to_club_value_eur").isNull().cast("int"))
    .withColumn("height_missing", F.col("height_in_cm").isNull().cast("int"))
    .withColumn("has_recent_appearance_history", F.col("has_recent_appearance_history").cast("int"))
)

print("Nulls before imputation - market_value:", data.filter(F.col("log_market_value").isNull()).count(),
      " from_club_value:", data.filter(F.col("log_from_club_value").isNull()).count(),
      " to_club_value:", data.filter(F.col("log_to_club_value").isNull()).count(),
      " height:", data.filter(F.col("height_in_cm").isNull()).count())

from_median = data.approxQuantile("log_from_club_value", [0.5], 0.01)[0]
to_median = data.approxQuantile("log_to_club_value", [0.5], 0.01)[0]
market_value_median = data.approxQuantile("log_market_value", [0.5], 0.01)[0]
height_median = data.approxQuantile("height_in_cm", [0.5], 0.01)[0]

data = (data
    .withColumn("log_from_club_value", F.coalesce("log_from_club_value", F.lit(from_median)))
    .withColumn("log_to_club_value", F.coalesce("log_to_club_value", F.lit(to_median)))
    .withColumn("log_market_value", F.coalesce("log_market_value", F.lit(market_value_median)))
    .withColumn("height_in_cm", F.coalesce("height_in_cm", F.lit(height_median)))
    .withColumn("sub_position", F.coalesce("sub_position", F.lit("Unknown")))
    .withColumn("foot", F.coalesce("foot", F.lit("Unknown")))
)
print("Medians used - from:", round(from_median, 2), " to:", round(to_median, 2),
      " market_value:", round(market_value_median, 2), " height:", round(height_median, 1))

## Temporal split, excluding the partial final year
Fees genuinely drift with market conditions (the COVID dip found earlier), so a random split would let the model train on 2024 deals and get evaluated on 2011 ones. 2026 is excluded entirely — the dataset freezes mid-year, so it's a structurally partial year, not a real full-year test.

In [0]:
train = data.filter(F.col("year") <= 2021)
val = data.filter((F.col("year") >= 2022) & (F.col("year") <= 2023))
test = data.filter((F.col("year") >= 2024) & (F.col("year") <= 2025))

print("train:", train.count(), " val:", val.count(), " test:", test.count())
print("Excluded as partial year:", data.filter(F.col("year") == 2026).count(), "rows from 2026")

In [0]:
league_counts = train.groupBy("to_domestic_competition_id").count()
rare_leagues = [row["to_domestic_competition_id"] for row in league_counts.filter(F.col("count") < 50).collect()]
print("Leagues bucketed as OTHER_OR_MISSING (<50 rows in train, or null):", rare_leagues)

def bucket_competition(df):
    return df.withColumn(
        "to_domestic_competition_id",
        F.when(F.col("to_domestic_competition_id").isin(rare_leagues) | F.col("to_domestic_competition_id").isNull(),
               F.lit("OTHER_OR_MISSING"))
         .otherwise(F.col("to_domestic_competition_id"))
    )

train = bucket_competition(train)
val = bucket_competition(val)
test = bucket_competition(test)

train.groupBy("to_domestic_competition_id").count().orderBy(F.desc("count")).show(30)

## Baseline: predicting the training mean
expm1 is applied to each individual prediction before computing the EUR-space error — never to an aggregate RMSE directly, since averaging-then-un-logging and un-logging-then-averaging give different, non-interchangeable numbers.

In [0]:
train_mean_log_fee = train.agg(F.avg("log_transfer_fee")).first()[0]
print("Training mean log fee:", train_mean_log_fee)

baseline_preds = test.withColumn("baseline_pred_log", F.lit(train_mean_log_fee)) \
    .withColumn("baseline_pred_eur", F.expm1("baseline_pred_log"))

baseline_rmse_log = baseline_preds.select(
    F.sqrt(F.avg(F.pow(F.col("log_transfer_fee") - F.col("baseline_pred_log"), 2)))
).first()[0]
baseline_mae_eur = baseline_preds.select(
    F.avg(F.abs(F.col("transfer_fee") - F.col("baseline_pred_eur")))
).first()[0]

print(f"Baseline RMSE (log space): {baseline_rmse_log:.4f}")
print(f"Baseline MAE (EUR space): €{baseline_mae_eur:,.0f}")

## Feature preparation
continuous_cols now includes age_squared and avg_minutes_per_appearance in place of minutes_last_365d, matching the diagnostic checks. OneHotEncoding is used for categoricals, not classification's VectorIndexer approach — a raw category index would be read as an ordinal number the model multiplies by a coefficient, meaningless for a nominal label like position. StandardScaler matters here specifically because regularization (Cell 6) penalizes coefficient size, only fair when features share a scale. This pipeline is fit on train only, unlike classification's indexers, since StandardScaler computes real statistics that val/test shouldn't influence. handleInvalid="keep" covers the risk of fitting on train alone — an unseen category in val/test gets bucketed as "unseen" instead of crashing.

In [0]:
categorical_cols = ["position", "sub_position", "foot", "to_domestic_competition_id"]
continuous_cols = ["age_at_transfer", "age_squared", "height_in_cm", "international_caps", "international_goals",
                    "log_market_value", "log_from_club_value", "log_to_club_value",
                    "goals_last_365d", "assists_last_365d", "avg_minutes_per_appearance", "appearances_last_365d"]
flag_cols = ["has_recent_appearance_history", "from_club_value_missing", "to_club_value_missing", "height_missing"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

continuous_assembler = VectorAssembler(inputCols=continuous_cols, outputCol="continuous_raw")
scaler = StandardScaler(inputCol="continuous_raw", outputCol="continuous_scaled", withMean=True, withStd=True)

final_assembler = VectorAssembler(
    inputCols=["continuous_scaled"] + [f"{c}_ohe" for c in categorical_cols] + flag_cols,
    outputCol="features"
)

prep_pipeline = Pipeline(stages=indexers + encoders + [continuous_assembler, scaler, final_assembler])
prep_model = prep_pipeline.fit(train)

def prep(df):
    return prep_model.transform(df)

train_prepped = prep(train)
val_prepped = prep(val)

## Regularization grid, selected on validation RMSE
elasticNetParam mixes L1 (pushes weak coefficients to exactly zero) and L2 (shrinks correlated features gracefully, relevant to the remaining moderate correlations like caps/goals or goals/assists) penalties.

In [0]:
evaluator_rmse = RegressionEvaluator(labelCol="log_transfer_fee", predictionCol="prediction", metricName="rmse")

best_rmse, best_params = float("inf"), None
for reg_param in [0.0, 0.01, 0.1]:
    for elastic_net in [0.0, 0.5, 1.0]:
        lr = LinearRegression(labelCol="log_transfer_fee", featuresCol="features",
                               regParam=reg_param, elasticNetParam=elastic_net)
        model = lr.fit(train_prepped)
        rmse = evaluator_rmse.evaluate(model.transform(val_prepped))
        print(f"regParam={reg_param}, elasticNetParam={elastic_net}: val RMSE={rmse:.4f}")
        if rmse < best_rmse:
            best_rmse, best_params = rmse, (reg_param, elastic_net)

print("Best params:", best_params, " val RMSE:", best_rmse)

## Final model — evaluated once on test, in both log space and real euros
R² is only reported in log space, since it doesn't have a coherent interpretation after a nonlinear back-transform.

In [0]:
train_val_prepped = prep(train.union(val))
test_prepped = prep(test)

final_lr = LinearRegression(labelCol="log_transfer_fee", featuresCol="features",
                             regParam=best_params[0], elasticNetParam=best_params[1])
final_model = final_lr.fit(train_val_prepped)
test_preds = final_model.transform(test_prepped)

evaluator_r2 = RegressionEvaluator(labelCol="log_transfer_fee", predictionCol="prediction", metricName="r2")

print("Test RMSE (log space):", evaluator_rmse.evaluate(test_preds))
print("Test R² (log space):", evaluator_r2.evaluate(test_preds))
print("Baseline RMSE (log space) for comparison:", baseline_rmse_log)

test_preds_eur = test_preds.withColumn("pred_fee_eur", F.expm1("prediction"))
test_mae_eur = test_preds_eur.select(F.avg(F.abs(F.col("transfer_fee") - F.col("pred_fee_eur")))).first()[0]
test_median_ape = test_preds_eur.select(
    F.expr("percentile_approx(abs(transfer_fee - pred_fee_eur) / transfer_fee, 0.5)")
).first()[0]

print(f"Test MAE (EUR space): €{test_mae_eur:,.0f}")
print(f"Baseline MAE (EUR space) for comparison: €{baseline_mae_eur:,.0f}")
print(f"Test median absolute percentage error: {test_median_ape:.1%}")

## Coefficients
Standardized continuous features make these roughly comparable. Reconstructing one-hot category names takes some bookkeeping, since the Pipeline splits indexing and encoding into separate stages.

In [0]:
coefficients = final_model.coefficients.toArray()

names = list(continuous_cols)
for c in categorical_cols:
    idx_stage = prep_model.stages[categorical_cols.index(c)]
    ohe_stage = prep_model.stages[len(categorical_cols) + categorical_cols.index(c)]

    n_categories_seen = ohe_stage.categorySizes[0]
    has_extra_bucket = n_categories_seen > len(idx_stage.labels)
    drop_last = ohe_stage.getDropLast()

    if has_extra_bucket:
        kept_labels = list(idx_stage.labels) if drop_last else list(idx_stage.labels) + ["UNSEEN"]
    else:
        kept_labels = list(idx_stage.labels[:-1]) if drop_last else list(idx_stage.labels)

    print(f"{c}: {len(idx_stage.labels)} labels, {n_categories_seen} categories seen, "
          f"{'extra unseen-value bucket present' if has_extra_bucket else 'clean'}")
    names += [f"{c}={lbl}" for lbl in kept_labels]
names += flag_cols

assert len(names) == len(coefficients), f"Mismatch: {len(names)} names vs {len(coefficients)} coefficients"

for name, coef in sorted(zip(names, coefficients), key=lambda x: -abs(x[1]))[:20]:
    print(f"{name}: {coef:+.4f}")